In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
df = pd.read_csv('../data/Processed_HR_Employee_Attrition.csv')

In [3]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, drop_first=True) # drop_first avoids multicollinearity

In [4]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. Feature Scaling (Fit on training data ONLY)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Initialize Model with Class Weighting
# The dataset is ~84% 'No' and 16% 'Yes'. 'balanced' adjusts weights inversely proportional to class frequencies.
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# 6. Cross-Validation Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_roc_auc = cross_val_score(log_reg, X_train_scaled, y_train, cv=cv, scoring='roc_auc')

print(f"Mean CV ROC-AUC: {cv_roc_auc.mean():.4f} (+/- {cv_roc_auc.std():.4f})")

# 7. Final Training and Prediction
log_reg.fit(X_train_scaled, y_train)
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

Mean CV ROC-AUC: 0.8326 (+/- 0.0328)


/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linea

In [5]:
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Test Set ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.81      0.87       247
           1       0.42      0.72      0.54        47

    accuracy                           0.80       294
   macro avg       0.68      0.77      0.70       294
weighted avg       0.86      0.80      0.82       294

Confusion Matrix:
 [[201  46]
 [ 13  34]]
Test Set ROC-AUC: 0.8298


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 1. Pipeline and K-Fold Setup
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Define metrics (Added 'accuracy')
scoring_metrics = ['f1', 'recall', 'precision', 'roc_auc', 'accuracy']

print("Executing 5-Fold Cross-Validation...\n")
cv_results = cross_validate(pipeline, X, y, cv=kf, scoring=scoring_metrics)

# 3. Formatted Output (Mean +/- Standard Deviation)
print("--- Logistic Regression Baseline: Class 1 (Attrition = Yes) ---")
print(f"F1-Score:  {np.mean(cv_results['test_f1']):.4f} (+/- {np.std(cv_results['test_f1']):.4f})")
print(f"Recall:    {np.mean(cv_results['test_recall']):.4f} (+/- {np.std(cv_results['test_recall']):.4f})")
print(f"Precision: {np.mean(cv_results['test_precision']):.4f} (+/- {np.std(cv_results['test_precision']):.4f})")
print(f"ROC-AUC:   {np.mean(cv_results['test_roc_auc']):.4f} (+/- {np.std(cv_results['test_roc_auc']):.4f})")
print(f"Accuracy:  {np.mean(cv_results['test_accuracy']):.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")

Executing 5-Fold Cross-Validation...

--- Logistic Regression Baseline: Class 1 (Attrition = Yes) ---
F1-Score:  0.5213 (+/- 0.0305)
Recall:    0.7592 (+/- 0.0407)
Precision: 0.3974 (+/- 0.0274)
ROC-AUC:   0.8376 (+/- 0.0197)
Accuracy:  0.7748 (+/- 0.0188)


/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linea